In [ ]:
import requests
import os
from io import BytesIO
from zipfile import ZipFile

In [ ]:
# Official page: https://gtfs.adelaidemetro.com.au/
# URLs
base_url = "https://gtfs.adelaidemetro.com.au/v1"
latest_version_number = "static/latest/version.txt"
latest_version_feed = "static/latest/google_transit.zip"

# binary data as response
request_headers={"Content-Type": "application/octet-stream"}

# Feed template
# practice: how to use string template + format
feed_url_template = "static/{version}/google_transit.zip"

# output
dest_path = 'destination'
os.makedirs(dest_path, exist_ok=True)

Read latest version

In [ ]:
# 1. Find the latest feed version
# 2. Download latest GTFS feed version using request

latest_feed_url = f"{base_url}/{latest_version_feed}"
response = requests.get(latest_feed_url, stream=True)

zip_path = f"{dest_path}/google_transit.zip"

with open(zip_path, "wb") as f:
    # stream the response body in small chunks instead of loading the whole file into memory at once
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)

print("Download complete!")

Dynamic read multiple feed versions: 930 to 939

In [ ]:
# 1. For loop the versions
# 2. Generate versioned URL
# 3. Request and handle response

for version in range(930, 940):
    print("Version: ", version)
    version_feed_url = f"{base_url}/static/{version}/google_transit.zip"
    print(version_feed_url)
    response = requests.get(version_feed_url, stream=True)

    zip_path = f"{dest_path}/google_transit_{version}.zip"
    print(zip_path)

    with open(zip_path, "wb") as f:
        # for chunk in response.iter_content(chunk_size=8192):
            f.write(response.content)

    print(f"Download version {version} complete!")

Unzip feed versions & export files

In [ ]:
# using ZipFile to read zipfile then extract all
import glob

# Define the path to your zip file
zip_files = glob.glob('destination/*.zip') 

# Define the directory where you want to extract the files (optional)
# If not specified, files will be extracted to the current working directory

for file_path in zip_files:
    extract_to_path = f'{file_path[:-4]}' 
    with ZipFile(file_path, 'r') as zip_obj:
        # Extract all files to the specified directory
        zip_obj.extractall(extract_to_path)
        print(f"Extract file {file_path} completed!")

Extra: extract zipfile from response.content

In [ ]:
# HINTS
# Use BytesIO (file bytes in memory) + ZipFile
# BytesIO create a memory buffer on RAM to store the response's content so that we don't have to create files on permanent memory

for version in range(930, 940):
    version_feed_url = f"{base_url}/static/{version}/google_transit.zip"
    response = requests.get(version_feed_url)

    extract_path = f"{dest_path}/google_transit_{version}"

    file = ZipFile(BytesIO(response.content))

    file.extractall(extract_path)

    print(f"Download and extract version {version} complete!")